# NovaHiring — Edge Cases del Chatbot

Este notebook prueba comportamientos especiales del sistema:

1. **Corrección de respuesta** — el candidato dice "me equivoqué" y el sistema re-hace la pregunta anterior
2. **Abandono de sesión** — el candidato dice "no quiero continuar" y el sistema cierra graciosamente
3. **Nueva sesión tras abandono** — verificar que el candidato puede volver a iniciar

## Requisitos
```bash
uv run fastapi dev main.py   # servidor corriendo
uv run python seed.py        # datos sembrados
```

> **Nota:** Este notebook crea y abandona sesiones de prueba para Elena Martínez.
> Ejecutarlo antes de `e2e_interview.ipynb` si quieres que el E2E corra limpio.

In [ ]:
import httpx
import json

BASE_URL     = 'http://localhost:8000'
JOB_ID       = 'job-clinica-salud-valencia-001'
CANDIDATE_ID = 'cand-11-elena-martinez'   # APTO — usamos Elena para los edge cases

client = httpx.Client(base_url=BASE_URL, timeout=60.0)

# Verificar servidor
try:
    client.get('/health').raise_for_status()
    print('✓ Servidor OK')
except httpx.ConnectError:
    print('✗ Servidor no está corriendo')
    raise SystemExit(1)


def start_session(candidate_id=CANDIDATE_ID):
    r = client.post('/api/v1/interviews/sessions', json={'job_id': JOB_ID, 'candidate_id': candidate_id})
    if r.status_code == 409:
        sid = r.json()['detail']['session_id']
        print(f'  ⚠ Sesión ya existe: {sid}')
        return sid
    r.raise_for_status()
    sid = r.json()['session_id']
    print(f'  ✓ Sesión creada: {sid}')
    return sid


def send(session_id, text):
    r = client.post(f'/api/v1/interviews/sessions/{session_id}/message', json={'content': text})
    if r.status_code == 410:
        print(f'  → 410 Sesión cerrada')
        return None
    r.raise_for_status()
    return r.json()


def get_state(session_id):
    return client.get(f'/api/v1/interviews/sessions/{session_id}').json()


print('✓ Helpers listos')

## Caso 1 — Corrección de respuesta anterior

Flujo:
1. Crear sesión → recibe pregunta D1
2. Responder D1 → recibe pregunta D2
3. Responder D2 → recibe pregunta D3
4. Enviar `"me equivoqué, quiero corregir mi respuesta"` → el sistema re-pregunta D2
5. Responder D2 de nuevo (respuesta corregida) → recibe pregunta D3 de nuevo
6. Abandonar la sesión

In [ ]:
print('=== CASO 1: Corrección de respuesta ===\n')

# 1. Crear sesión
sid = start_session()
state = get_state(sid)
print(f'  Estado inicial: pregunta {state["current_question_index"] + 1}/8')

# 2. Responder D1
print('\n→ Enviando respuesta D1...')
resp = send(sid, 'He integrado WhatsApp Business API directamente con Meta Cloud API en un proyecto de e-commerce.')
print(f'  Siguiente pregunta: {resp["next_question"]["dimension_id"]} — {resp["next_question"]["dimension_name"]}')
print(f'  Estado sesión: {resp["session_status"]}')

# 3. Responder D2
print('\n→ Enviando respuesta D2...')
resp = send(sid, 'Implementé cifrado AES-256 para datos sanitarios y el RAT con asesoría jurídica.')
print(f'  Siguiente pregunta: {resp["next_question"]["dimension_id"]} — {resp["next_question"]["dimension_name"]}')

state = get_state(sid)
print(f'\n  Estado antes de corregir: current_question_index = {state["current_question_index"]} (sobre D3)')

In [ ]:
# 4. Intentar corregir la respuesta anterior (D2)
print('→ Enviando petición de corrección...')
resp = send(sid, 'me equivoqué, quiero corregir mi respuesta anterior, fue incompleta')

next_q = resp['next_question']
print(f'\n  ✓ El sistema re-pregunta: {next_q["dimension_id"]} — {next_q["dimension_name"]}')
print(f'  Pregunta #{next_q["question_number"]} de {next_q["total_questions"]}')
print(f'  Estado sesión: {resp["session_status"]}')

state = get_state(sid)
print(f'\n  current_question_index = {state["current_question_index"]} (volvió a D2 ✓)')

assert next_q['dimension_id'] == 'D2', f'Esperado D2, obtenido {next_q["dimension_id"]}'
print('  ✓ Assertion correcta: el sistema re-preguntó D2')

In [ ]:
# 5. Responder D2 con respuesta corregida
print('→ Enviando respuesta corregida para D2...')
resp = send(sid, 'Implementé cifrado AES-256, consentimiento explícito con registro de auditoría, RAT con validación del DPO y cifrado en tránsito TLS 1.3.')

next_q = resp['next_question']
print(f'  ✓ Siguiente pregunta: {next_q["dimension_id"]} — {next_q["dimension_name"]}')
print(f'  El sistema avanza a D3 correctamente ✓')

state = get_state(sid)
print(f'  current_question_index = {state["current_question_index"]} (apunta a D3 ✓)')

## Caso 2 — Abandono de sesión

Flujo:
1. (Misma sesión, estamos en D3)
2. Enviar `"no quiero continuar"` → sistema cierra graciosamente con mensaje de despedida
3. Verificar que la sesión queda en estado `abandoned`
4. Intentar enviar otro mensaje → `410 Gone`

In [ ]:
print('=== CASO 2: Abandono de sesión ===\n')
print(f'  Sesión activa: {sid}')

# Abandonar
print('\n→ Enviando "no quiero continuar"...')
resp = send(sid, 'no quiero continuar con la entrevista')

print(f'  Estado sesión: {resp["session_status"]}')
print(f'  Siguiente pregunta: {resp["next_question"]}  (debe ser None ✓)')
print(f'  Resultado evaluación: {resp["evaluation_result"]}  (debe ser None ✓)')

# Verificar en DB via API
state = get_state(sid)
print(f'\n  Estado en DB: {state["status"]}')

assert resp['session_status'] == 'abandoned', f'Esperado abandoned, obtenido {resp["session_status"]}'
assert state['status'] == 'abandoned'
print('  ✓ Sesión correctamente marcada como abandoned')

In [ ]:
# Intentar enviar otro mensaje → debe recibir 410
print('→ Intentando enviar mensaje a sesión abandonada...')
try:
    r = client.post(f'/api/v1/interviews/sessions/{sid}/message', json={'content': 'hola?'})
    r.raise_for_status()
    print('  ✗ ERROR: no debería haber aceptado el mensaje')
except httpx.HTTPStatusError as e:
    print(f'  ✓ Recibió {e.response.status_code} — {e.response.json()["detail"]["error"]}')
    assert e.response.status_code == 410

## Caso 3 — Nueva sesión tras abandono

Un candidato que abandonó debe poder iniciar una nueva sesión (en la vida real esto requeriría aprobación del reclutador, pero técnicamente debe funcionar).

In [ ]:
print('=== CASO 3: Nueva sesión tras abandono ===\n')

# Crear nueva sesión para el mismo candidato
print('→ Creando nueva sesión para Elena Martínez (misma que abandonó)...')
r = client.post('/api/v1/interviews/sessions', json={'job_id': JOB_ID, 'candidate_id': CANDIDATE_ID})

print(f'  HTTP {r.status_code}')
assert r.status_code == 201, f'Esperado 201, obtenido {r.status_code}: {r.text}'

new_sid = r.json()['session_id']
print(f'  ✓ Nueva sesión creada: {new_sid}')
print(f'  (anterior abandonada: {sid})')
assert new_sid != sid

# Limpiar: abandonar esta también para no afectar el E2E notebook
print('\n→ Abandonando la nueva sesión (limpieza para no bloquear e2e_interview.ipynb)...')
resp = send(new_sid, 'no quiero continuar con la entrevista')
print(f'  Estado: {resp["session_status"]} ✓')

In [ ]:
print('\n' + '='*55)
print('RESUMEN — Edge Cases')
print('='*55)
print('✓ Caso 1: Corrección de respuesta anterior — OK')
print('          El sistema re-preguntó D2 al detectar "me equivoqué"')
print('✓ Caso 2: Abandono de sesión — OK')
print('          Sesión marcada como "abandoned", próximo mensaje recibe 410')
print('✓ Caso 3: Nueva sesión tras abandono — OK')
print('          El candidato puede iniciar una sesión nueva')
print()
print('Las sesiones de prueba quedan en estado "abandoned".')
print('Puedes correr e2e_interview.ipynb normalmente — el 409 handler')
print('detectará que no hay sesión activa y creará una nueva.')